# TactileLens Track C: partial fine-tuning of PaddleOCR-VL on Colab (LoRA): NOT RUN

**This notebook has never been run.** Every number it prints is a measurement of *your* run; nothing
here is a result yet. You sign in to Google and press every Run button yourself.

What is documented (checked 2026-11-07; sources in `docs/training_runbook.md`):

- ERNIEKit's guide fine-tunes **`PaddlePaddle/PaddleOCR-VL` (the original 0.9B)**, full fine-tuning,
  example config `examples/configs/PaddleOCR-VL/sft/run_ocr_vl_sft_16k.yaml` (bf16, flash attention,
  recompute). **PaddleOCR-VL-1.6 is not named anywhere as a fine-tuning target**, so this notebook
  trains the 0.9B model and refuses 1.6 unless you change `MODEL_ID` knowingly (recorded in `run.json`).
- **LoRA**: an ERNIE maintainer states that `fine_tuning=LoRA` with `lora_rank` starts LoRA SFT and
  that learning rate should be raised (ERNIE issue #1397). There is no shipped LoRA config, and
  exporting/merging a LoRA model had an unanswered error report there. Both are **not verified**.
- **Freezing the vision encoder / training only some layers: not documented** in ERNIEKit or
  PaddleFormers for PaddleOCR-VL. Not used here.
- The app's online engine (Baidu's hosted API) cannot be fine-tuned and is not changed by this.

GPU: the documented config needs **bf16 + flash attention (compute capability >= 8.0)**. The free
Colab GPU is a **T4 (7.5)**, so by default cell 1 **stops on a T4**. `EXPERIMENTAL_T4 = True` runs an
undocumented fp16 / no-flash-attention variant as a **20-step feasibility and memory test only**.

Upload only our own images (no textbook or third-party pages, no personal data). The eval split is
never given to training.

### Google Drive layout (create `MyDrive/tactilelens/` once)

```
MyDrive/tactilelens/
  tools/training_vl/          convert_to_sft.py, vl_common.py, config.json,
                              evaluate_vl.py, local_vl_worker.py      (~0.1 MB, from the repo)
  data/<split id>.zip         images/ + labels.csv from tools/dataset/split_dataset.py
  cache/PaddleOCR-VL/         base model, downloaded once by cell 5   (~2.0 GiB)
  runs/<run id>/              run.json, logs/, checkpoints/ (resume), adapter or merged model, eval/
```

In [ ]:
# 1. GPU check. Stops on GPUs the documented config does not support.
import subprocess
info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.strip()
if not info:
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU.")
print(info)
GPU_NAME, GPU_MIB, GPU_CC = [s.strip() for s in info.splitlines()[0].split(",")]
GPU_MIB, GPU_CC = int(GPU_MIB), float(GPU_CC)

EXPERIMENTAL_T4 = False   # True = undocumented fp16 test on a T4, 20 steps, memory/feasibility only
MIN_GIB = 14              # below this even LoRA with batch 1 is not worth trying

if GPU_MIB < MIN_GIB * 1024:
    raise SystemExit(f"{GPU_NAME}: {GPU_MIB // 1024} GiB is too small (need >= {MIN_GIB} GiB).")
if GPU_CC < 8.0 and not EXPERIMENTAL_T4:
    raise SystemExit(f"{GPU_NAME} (compute capability {GPU_CC}) has no bf16 / flash attention, which the "
                     "documented config uses. Stop here, or set EXPERIMENTAL_T4 = True for a 20-step test.")
MODE = "documented (bf16)" if GPU_CC >= 8.0 else "EXPERIMENTAL fp16 test (undocumented)"
print("Mode:", MODE)

In [ ]:
# 2. Settings and Google Drive (you sign in when asked).
from google.colab import drive
drive.mount("/content/drive")

import json, time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/tactilelens")
SPLIT_ZIP = BASE / "data" / "CHANGE_ME_split_id.zip"     # from split_dataset.py
MODEL_ID = "PaddlePaddle/PaddleOCR-VL"                    # the only documented fine-tuning target
ERNIE_REF = "develop"     # replace with the commit hash printed in cell 4 after the first run

LORA_RANK = 8
LEARNING_RATE = 1e-4      # full SFT example uses 5e-6; LoRA needs more (maintainer). A starting guess.
EPOCHS = 2
GRAD_ACCUMULATION = 8
MAX_SEQ_LEN = 4096        # formula / short text crops: about 150-1,300 image tokens each
SAVE_STEPS = 50           # checkpoint to Drive this often (resume after a timeout)
TEST_STEPS = 20           # EXPERIMENTAL_T4 runs only this many steps

if MODEL_ID != "PaddlePaddle/PaddleOCR-VL":
    print(f"WARNING: {MODEL_ID} is not a documented fine-tuning target. This is an experiment.")

# Resume: reuse the last run unless it finished.
current = BASE / "runs" / "CURRENT_RUN.txt"
RUN_ID = current.read_text().strip() if current.exists() else time.strftime("%Y%m%d_%H%M%S")
RUN = BASE / "runs" / RUN_ID
(RUN / "logs").mkdir(parents=True, exist_ok=True)
current.write_text(RUN_ID)
print("Run folder:", RUN)

In [ ]:
# 3. Data: unpack the split, convert with convert_to_sft.py, check the eval guard again.
import hashlib, shutil, subprocess, sys, zipfile
DATA = Path("/content/data")
if not (DATA / "labels.csv").exists():
    with zipfile.ZipFile(SPLIT_ZIP) as archive:
        archive.extractall(DATA)
TOOLS = Path("/content/training_vl")
shutil.copytree(BASE / "tools" / "training_vl", TOOLS, dirs_exist_ok=True)
# The PC already checked the recognizer's eval sets in split_dataset.py; they are not on Colab.
result = subprocess.run([sys.executable, "convert_to_sft.py", "--data", str(DATA / "images"),
                         "--labels", str(DATA / "labels.csv"), "--out", str(DATA / "sft"),
                         "--absolute-urls", "--skip-recognizer-eval-check"],
                        cwd=TOOLS, capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-2000:])
if result.returncode:
    raise SystemExit("convert_to_sft.py refused the data (see above). Nothing trained.")

def meta(split):
    path = DATA / "sft" / f"{split}_meta.jsonl"
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
eval_sha = {m["sha256"] for m in meta("eval")}
for split in ("train", "val"):
    for m in meta(split):
        assert m["sha256"] not in eval_sha, f"LEAK: {m['filename']}"
N_TRAIN, N_VAL, N_EVAL = len(meta("train")), len(meta("val")), len(eval_sha)
print(f"train {N_TRAIN}, val {N_VAL}, eval {N_EVAL} (held out, never trained on)")

In [ ]:
# 4. Environment: PaddlePaddle GPU and ERNIEKit, as in the ERNIEKit guide (CUDA 12.6 wheel).
# Sizes are not known to us in advance; Colab downloads them, not your PC.
!python -m pip install -q paddlepaddle-gpu==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -c "import paddle; paddle.utils.run_check(); print(paddle.__version__)"
!test -d /content/ERNIE || git clone -q https://github.com/PaddlePaddle/ERNIE /content/ERNIE
!cd /content/ERNIE && git checkout -q {ERNIE_REF} && git rev-parse HEAD
!cd /content/ERNIE && python -m pip install -q -r requirements/gpu/requirements.txt && python -m pip install -q -e .

In [ ]:
# 5. Base model, cached on Drive so later sessions do not download it again (~2.0 GiB once).
MODEL_DIR = BASE / "cache" / MODEL_ID.split("/")[-1]
if not (MODEL_DIR / "model.safetensors").exists():
    !huggingface-cli download {MODEL_ID} --local-dir "{MODEL_DIR}"
print(sorted(p.name for p in MODEL_DIR.iterdir())[:12])

In [ ]:
# 6. Train (LoRA). Checkpoints go straight to Drive; after a timeout, run cells 1-5 again and this
#    cell resumes from the newest checkpoint (resume_from_checkpoint: not verified for ERNIEKit).
import math, re
CKPT = RUN / "checkpoints"
CKPT.mkdir(exist_ok=True)
done = sorted(CKPT.glob("checkpoint-*"), key=lambda p: int(re.sub(r"\D", "", p.name) or 0))
resume = f"resume_from_checkpoint={done[-1]}" if done else ""
max_steps = TEST_STEPS if GPU_CC < 8.0 else max(1, math.ceil(N_TRAIN * EPOCHS / GRAD_ACCUMULATION))
precision = ("compute_type=bf16 use_flash_attention=True" if GPU_CC >= 8.0 else
             "compute_type=fp16 use_flash_attention=False use_sparse_flash_attn=False")
run_info = {"run_id": RUN_ID, "mode": MODE, "gpu": GPU_NAME, "gpu_mib": GPU_MIB, "model": MODEL_ID,
            "ernie_ref": ERNIE_REF, "lora_rank": LORA_RANK, "lr": LEARNING_RATE, "max_steps": max_steps,
            "max_seq_len": MAX_SEQ_LEN, "train": N_TRAIN, "val": N_VAL, "eval_held_out": N_EVAL,
            "resumed_from": str(done[-1]) if done else None, "started": time.strftime("%Y-%m-%d %H:%M:%S")}
(RUN / "run.json").write_text(json.dumps(run_info, indent=1))
print(run_info)

!nvidia-smi --query-gpu=memory.used --format=csv,noheader -l 5 > "{RUN}/logs/gpu_memory.csv" 2>&1 &
!cd /content/ERNIE && CUDA_VISIBLE_DEVICES=0 erniekit train examples/configs/PaddleOCR-VL/sft/run_ocr_vl_sft_16k.yaml \
    model_name_or_path="{MODEL_DIR}" \
    train_dataset_path=/content/data/sft/train.jsonl eval_dataset_path=/content/data/sft/val.jsonl \
    fine_tuning=LoRA lora_rank={LORA_RANK} learning_rate={LEARNING_RATE} {precision} \
    max_steps={max_steps} warmup_steps={max(1, max_steps // 100)} gradient_accumulation_steps={GRAD_ACCUMULATION} \
    max_seq_len={MAX_SEQ_LEN} save_steps={SAVE_STEPS} save_total_limit=3 \
    output_dir="{CKPT}" logging_dir="{RUN}/logs/tensorboard" {resume} 2>&1 | tee -a "{RUN}/logs/train.log"
!pkill -f "nvidia-smi --query-gpu" || true
peak = max((int(l.split()[0]) for l in open(RUN / "logs" / "gpu_memory.csv") if l.split() and l.split()[0].isdigit()),
           default=None)
run_info.update({"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu_peak_mib_nvidia_smi": peak})
(RUN / "run.json").write_text(json.dumps(run_info, indent=1))
print("GPU peak (nvidia-smi, MiB):", peak, "| see logs/train.log for errors before reading anything into it")

In [ ]:
# 7. Merge LoRA into a full model folder PaddleOCR can load (NOT VERIFIED: an export error was reported
#    upstream). If this fails, stop and report the error; the adapter checkpoints stay on Drive.
MERGED = RUN / "merged"
!cd /content/ERNIE && erniekit export examples/configs/PaddleOCR-VL/sft/run_ocr_vl_sft_16k.yaml \
    model_name_or_path="{MODEL_DIR}" output_dir="{CKPT}" lora=True 2>&1 | tee "{RUN}/logs/export.log"
print(sorted(p.name for p in CKPT.iterdir()))
print("Copy the merged weights to", MERGED, "once the export output folder is known from the log above.")

In [ ]:
# 8. Evaluate on the held-out eval set with evaluate_vl.py (needs paddleocr in this runtime).
!python -m pip install -q "paddleocr[doc-parser]"
for f in ("chat_template.jinja", "inference.yml"):     # as the ERNIEKit guide says
    if (MODEL_DIR / f).exists() and MERGED.exists():
        shutil.copy(MODEL_DIR / f, MERGED / f)
import sys
!cd /content/training_vl && python evaluate_vl.py --eval /content/data/sft --engine finetuned \
    --model-dir "{MERGED}" --vl-model-name PaddleOCR-VL-0.9B --pipeline-version v1 --vl-python {sys.executable} \
    2>&1 | tee "{RUN}/logs/eval_finetuned.log"
!cd /content/training_vl && python evaluate_vl.py --eval /content/data/sft --engine finetuned \
    --model-dir "{MODEL_DIR}" --vl-model-name PaddleOCR-VL-0.9B --pipeline-version v1 --vl-python {sys.executable} \
    2>&1 | tee "{RUN}/logs/eval_base_0.9B.log"
print("Compare the two logs: same eval set, base 0.9B vs fine-tuned. Report only these measured numbers.")

## 9. Download the result

`runs/<run id>/` on Drive holds `run.json`, `logs/` (train, GPU memory, export, both evaluations) and
the checkpoints / merged model. Download `merged/` (about 1.8 GiB) to the PC only if the evaluation
shows a clear gain over the base 0.9B on the held-out set, then follow "Serving for the demo" in
`docs/training_runbook.md`. When the run is finished, delete `runs/CURRENT_RUN.txt` so the next run
starts fresh.